#### A. Librerías, variables, funciones y lectura de datos.

In [ ]:
#1. Importo librerías.
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
import holidays
from scipy import signal

In [ ]:
#2. Lectura de datos.
series_temporales = pd.read_csv("../dataset.csv")
diccionario = pd.read_excel("../diccionario.xlsx",sheet_name="metadata")

#### B. EDA.

In [ ]:
#1. Columnas y variables.
print("Tenemos {} columnas. Estas son: {}".format(series_temporales.shape[1],series_temporales.columns))

In [ ]:
#2. Información general de las variables.
series_temporales.info() 

# Tenemos series temporales con datos nulos......

In [ ]:
#3. Convertimos tipo de datos de la fecha a datetime y la ordenamos por dicha variable.
series_temporales["fecha"] = pd.to_datetime(series_temporales["fecha"])
series_temporales = series_temporales.sort_values("fecha")

In [ ]:
#4. Borramos filas enteras con NA.
series_temporales.dropna(how='all', inplace=True)

In [ ]:
#5. Entendemos las fechas de inicio y fin de la Serie temporal
inicio = series_temporales["fecha"].sort_values(ascending=True).iloc[0]
fin = series_temporales["fecha"].sort_values(ascending=False).iloc[0]

print(f"Las series temporales son algunas diarias, y otras mensuales, yendo desde {inicio} a {fin}.")

In [ ]:
#6. Nos quedamos con la información de 01/2025 hasta 04/2025 (coincidente con las fechas de las noticias screappeadas).
series_temporales = series_temporales[(series_temporales["fecha"] >= "2025-01-01")&(series_temporales["fecha"] <= "2025-04-30")]

In [ ]:
#7. Tiene fechas repetidas?
print("Fechas unicas: {}".format(series_temporales["fecha"].nunique()))
print("Cantidad de filas totales: {}".format(series_temporales.shape[0]))

In [ ]:
#8. Entre el inicio y fin hay 79 fechas?
inicio = series_temporales["fecha"].sort_values(ascending=True).iloc[0]
fin = series_temporales["fecha"].sort_values(ascending=False).iloc[0]

print("Tenemos una apertura de fechas de {} días".format(fin - inicio))
print("Debería cubrir {} días".format(series_temporales.shape[0]))

In [ ]:
#9. Que fechas faltan?
#a. Análisis general.
inicio = series_temporales["fecha"].min()
fin = series_temporales["fecha"].max()

rango_completo = pd.date_range(start=inicio, end=fin, freq='D')

fechas_reales = series_temporales["fecha"].unique()

fechas_faltantes = rango_completo.difference(fechas_reales)

#print("Cantidad de fechas faltantes:", len(fechas_faltantes))
#print("Fechas faltantes:")
#print(fechas_faltantes)
#print("-------------------------------------------------------------------")
#b. Análisis de días de la semana.
faltantes_df = pd.DataFrame(fechas_faltantes, columns=["fecha"])
faltantes_df["dia_semana"] = faltantes_df["fecha"].dt.day_name(locale="es_ES")
faltantes_laborales = faltantes_df[
    ~faltantes_df["dia_semana"].isin(["Sábado", "Domingo"])
]

#print("Cantidad de fechas faltantes en días hábiles:", len(faltantes_laborales))
#print(faltantes_laborales)
#print("-------------------------------------------------------------------")

#c. Excluimos los feriados nacionales.
feriados_ar = holidays.AR(years=range(inicio.year, fin.year + 1))
faltantes_laborales["es_feriado"] = faltantes_laborales["fecha"].isin(feriados_ar)
faltantes_no_fds_ni_feriados = faltantes_laborales[~(faltantes_laborales["es_feriado"])]

print("Cantidad de fechas faltantes que no son fin de semana ni feriados:", len(faltantes_no_fds_ni_feriados))
print(faltantes_no_fds_ni_feriados)

In [ ]:
#10. Analisis estadísticos descriptivos básicos de las series temporales.
series_temporales.describe()

#### C. Selección de variables (Feature Selection).

##### 1. Análisis estadístico.

In [ ]:
#1. Análisis estadístico previo.
#a. Eliminamos la columna 'fecha' y calculamos la matriz de correlaciones.
corr_matrix = series_temporales.drop(columns=['fecha']).corr()

#b. Mostramos la matriz.
print(corr_matrix)

#c. visualizamos con un mapa de calor.
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, cbar_kws={'shrink': .8})
plt.title('Matriz de correlaciones entre variables')
plt.show()

##### 2. Análisis conceptual.

In [ ]:
#1. Con el único precio del Merval que me voy a quedar es con el de apertura, ya que sería Data Leakeage utilizar las demás.
series_temporales.drop(["merval_maximo","merval_minimo","merval_cierre"],axis=1,inplace=True)

In [ ]:
#2. Mercado financiero local e internacional.
# embi_spread_arg → mide riesgo país argentino, impacta directamente en valuaciones y flujos hacia equity local.
# embi_spreads_brz → referencia regional, ayuda a distinguir si el movimiento es idiosincrático argentino o regional.
# embi_spread_global → mide apetito global por riesgo; buen predictor en contextos de estrés global.

In [ ]:
#3. Tipo de cambio y reservas.
# tc_mayorista / tc_minorista → el tipo de cambio suele anticipar o acompañar movimientos del equity argentino (devaluación = caída en USD, pero subas nominales en ARS).
# rrii (reservas internacionales) → indicador de fortaleza macro, política monetaria y confianza.

In [ ]:
#4. Tasas e instrumentos financieros ----> reflejan el costo del dinero. Cambios en tasas reales afectan valuación de acciones (vía DCF y flujos futuros).
# Vamos a elegir solamente BADLAR para evitar multicolinealidad.
#series_temporales.drop(["tamar","tm20", "tasa_pf_pesos", "tasa_pf_dolares","tasa_prestamos_personales"],axis=1,inplace=True)
series_temporales.drop(["tamar","tm20","tasa_prestamos_personales"],axis=1,inplace=True)

In [ ]:
#5. Variables monetarias.
# base_monetaria, m1, m2_transaccional → reflejan liquidez del sistema. En Argentina, expansión monetaria suele correlacionar con subas nominales del Merval (no reales).
# Se descartan dep_ccorrientes, dep_cahorro, dep_plazo, dep_plazo_fijo, dep_dolar_priv_ars ya que todas son componentes de M1 o M2. Y m2 ya que está muy correlacionada con m2_transaccional.
series_temporales.drop(["dep_ccorrientes", "dep_cahorro", "dep_plazo", "dep_plazo_fijo", "dep_dolar_priv_ars","m2"],axis=1,inplace=True)

In [ ]:
#6. Actividad real.
# emae_tendencia_ciclo → indicador del ciclo económico real argentino.
# Se descartan emae_original y emae_desestacionalizado para evitar colinealidad. 
# Además, emae_tendencia_ciclo captura la tendencia de largo plazo y el ciclo económico subyacente, sin el ruido diario/mensual. 
# Es más estable y predictivo para un índice bursátil que responde a expectativas macro, no a movimientos mensuales aislados.
series_temporales.drop(["emae_original","emae_desestacionalizado"],axis=1,inplace=True)

In [ ]:
#7. Expectativas e inflación.
# inflación: importante para variables nominales (Merval en ARS).
# REM: anticipa expectativas.
# CER/ UVA: capturan indexación y costo de financiamiento ----> Vamos a descatarlas porque aportan información derivada de inflacion (el efecto conceptual ya lo estamos capturando).
series_temporales.drop(["cer","uva"],axis=1,inplace=True)

In [ ]:
#8. Préstamos.
# Me quedo solamente con prestamos_privados_pesos_ars.
# Se descarta prestamos_privados_dolares_ars y prestamos_privados_ars porque están muy correlacionados.
series_temporales.drop(["prestamos_privados_dolares_ars","prestamos_privados_ars"],axis=1,inplace=True)

In [ ]:
print("En total me quedo con {} columnas regresoras para predecir el precio de apertura del Merval.".format(len(series_temporales.columns)-2))

#### E. Análisis básicos en el Dominio del Tiempo, y refinamiento del dataset.

In [ ]:
#1. Corregimos datos incorrectos.
#a. Merval 24/03/2024 fue feriado, se elimina la fila (el dato además de ser outlier es incorecto).
series_temporales = series_temporales[series_temporales["fecha"] != "2024-03-24"]

In [ ]:
#2. Imputamos los datos faltantes --> Las ST para que funcione la TTF requiere que el delta t de las muestras sea constante. Además, posteriormente para trabajar con dos series temporales juntas vamos a necesitar que compartan los mismos intervalos de muestreo (1 dato por día).
for _, row in diccionario.iterrows():
    var = row['serie']
    freq = row['frecuencia']
    
    #a. Salteo si la variable no existe en el DataFrame (variable vieja o ya borrada/trabajada).
    if var not in series_temporales.columns:
        print(f"Advertencia: la variable '{var}' no existe. Se saltea.")
        continue

    #b. Merval.
    if var.lower() == 'merval':
        continue  # no tocamos Merval ya que es la variable a predecir, y no queremos "inventar" valores.
    
    #c. Variables diarias → interpolación lineal.
    if freq == 'diaria':
        series_temporales[var] = series_temporales[var].interpolate(method='linear')
    
    #d. Variables mensuales → forward fill (tomo el último dato).
    elif freq == 'mensual':
        series_temporales[var] = series_temporales[var].ffill()

    #e. Si aún quedan NaN al inicio (sin dato anterior), completamos hacia atrás (pasa con el primer dato de embi_spreads_brz y de embi_spread_global).
    series_temporales[var] = series_temporales[var].bfill()

In [ ]:
#3. Graficamos las columnas numéricas.
#a. Listamos las columnas numéricas (excluyendo 'fecha').
variables = series_temporales.drop(columns=["fecha"]).columns

#b. Graficamos cada variable en un gráfico separado.
for var in variables:
    plt.figure(figsize=(12,5))  # Nueva figura para cada variable
    plt.plot(series_temporales["fecha"], series_temporales[var], marker='.')
    plt.title(f"Evolución de {var} en el tiempo")
    plt.xlabel("Fecha")
    plt.ylabel(var)
    plt.grid(True)
    plt.show()

In [ ]:
#4. Primer análisis de la ST del Merval.
# A simple vista, parece vislumbrarse una tendencia general alcista, teniendo en cuenta la totalidad del período analizado. Este período comienza especialmente con las elecciones PASO, primera vuelta y ballotage que resulta como presidente Milei.
# Luego de un período lentamente ascendente desde 09/2023,se distinguen distintos ciclos de frecuencias más altas, pero que tampoco parecen ser mensuales ni de cierta temporalización común.
# También cabe destacar un período de gran pendiente alcista entre 10/2024 y 01/2025, momento donde la serie alcanza un pico máximo en principios del 2025, y una posterior tendencia bajista en los meses posteriores. Como hipótesis suponemos la existencia de algún evento económico y/o político puntual (corrupción? suba del dolar? elecciones bonarenses?).
# El final de esta serie, no está excepta de estos ciclos con picos locales y valles a sus costados. Aún más, parece apreciarse mayor volatilidad en el precio de apertura del Merval. 
# Además, durante toda la serie se puede observar cierto ruido de frecuencias altas, posiblemente a cambios semanales/diarios. Y la influencia de variables predictoras.

# Acompaña nuestra hipótesis que el Riesgo País tiene una tendencia bajista. Esto también se ve en el período posterior a 01/2025 donde, mientras el Merval desciende, el Riesgo País aumenta.
# Si analizamos el posible impacto del EMAE (actividad económica), podemos ver que desde Mayo/2024 se ve una tendencia a la recuperación, que se ve correspondida con los datos del Merval desfasados. Podemos decir que el Merval estaría "explicando" al EMAE. Cabe aclarar que este tipo de indicadores suelen tener una ventana temporal para verse su impacto, como así también podemos pensar que el Merval se anticipó a partir de las expectativas generadas sobre el EMAE.
# La mayor volatilidad (frecuencias más altas) que observamos a partir de 2025 también se observa en las series temporales del tipo de cambio minorista y mayorista. Sin dudas acá puede haber una explicación del fenómeno.
# Con respecto a las reservas internacionales, dede 01/2024 se puede vislumbrar una tendencia alcista de forma volatil. Dichos cambios no se ven reflejados en el precio de apertura del Merval.
# Respecto al contexto de base monetaria, circulación monetaria, y prestámos privados en pesos, hay una tendencia general alcista, con diferentes ciclos de distintas frecuencias al interior de cada una de las series aquí planteadas.
# También podemos ver una inflación mensual con una tendencia bajista, aunque con ciertos vaivenes propios de la economía doméstica. Este dato viene acompañado por una inflación esperada (rem) también bajista, lo que nos habla de las expectativas en esta materia.

#### 5. Exportación.

In [90]:
#a. Elimino variables post análisis que veo que no me van a servir.
series_temporales.drop(["rrii","embi_spreads_brz"],axis=1,inplace=True)

In [91]:
#b. Exporto.
series_temporales.to_csv("../dataset_var_econom_limpio.csv",index=False)